## 第5章 解析和生成器

### 1.集合处理内置函数

- `map(function, iterables, ...)`：返回迭代器，将函数(function)依次应用到可迭代对象(iterables)的每个元素。传入多个可迭代对象时，按最短的可迭代对象长度停止迭代。

In [ ]:
# [(0, 'A'), (1, 'B'), (2, 'C'), (3, 'D')]
print(list(map(lambda *x: x, range(5), 'ABCD')))

# [(0, 'A'), (1, 'B'), (2, 'C')]
# 因为'ABC'的长度小于'1234'，所以只迭代到'ABC'的长度
print(list(map(lambda *x: x, range(4), 'ABC')))

- `zip(iterables, ...)`：返回迭代器，将多个可迭代对象的第i个元素配成元组，按最短的可迭代对象长度停止。如果指定`strict=True`，则在最短可迭代对象耗尽后，触发`StopIteration`异常。

In [ ]:
grades = [18, 23, 30, 27]
avgs = [22, 21, 29, 24]

# [(18, 22), (23, 21), (30, 29), (27, 24)]
print(list(zip(grades, avgs)))
print(list(map(lambda *x: x, grades, avgs)))

a = [5, 9, 2, 4, 7]
b = [3, 7, 1, 9, 2]
c = [6, 8, 0, 5, 3]

# [6, 9, 2, 9, 7]
print(list(map(lambda x: max(x), zip(a, b, c))))
print(list(map(lambda *x: max(x), a, b, c)))

- `filter(function, iterable)`：返回迭代器，将函数(function)依次应用到可迭代对象(iterable)的每个元素，返回结果为True的元素。若function为None，则过滤掉所有为假的元素。

In [ ]:
test = [2, 5, 8, 0, 0, 1, 0]

# [2, 5, 8, 1]
print(list(filter(None, test)))

# [5, 8]
print(list(filter(lambda x: x > 4, test)))

### 2.推导式

- 列表推导式：`[expression for item in iterable if condition]`。

In [ ]:
# [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
print([x ** 2 for x in range(10)])
print(list(map(lambda x: x **2, range(10))))

# [0, 4, 16, 36, 64]
print([x ** 2 for x in range(10) if not x % 2])
print(list(map(lambda x: x ** 2, filter(lambda x: not x % 2, range(10)))))

In [ ]:
# 求直角边均小于10的所有直角三角形
from math import sqrt

mx = 10

# 通过filter函数筛选出直角三角形
# 输出：[(3, 4, 5.0), (6, 8, 10.0)]
triples = [(a, b, sqrt(a**2 + b**2)) for a in range(1, mx) for b in range(a, mx)]
triples = list(filter(lambda x: x[2].is_integer(), triples))  # 筛选出斜边为整数的直角三角形
print(triples)

# 通过map函数将直角三角形的斜边转换为整数
# 输出：[(3, 4, 5), (6, 8, 10)]
triples = [(a, b, sqrt(a**2 + b**2)) for a in range(1, mx) for b in range(a, mx)]
triples = list(filter(lambda x: x[2].is_integer(), triples))
triples = list(map(lambda x: x[:2]+(int(x[2]),), triples))  # 利用元组相加
print(triples)

# 通过列表推导式将直角三角形的斜边转换为整数
# 输出：[(3, 4, 5), (6, 8, 10)]
triples = [(a, b, sqrt(a**2 + b**2)) for a in range(1, mx) for b in range(a, mx)]
triples = [(a, b, int(c)) for a, b, c in triples if c.is_integer()]
print(triples)

# 通过列表推导式一次性实现筛选和转换操作
# 输出：[(3, 4, 5), (6, 8, 10)]
triples = [(a, b, int(c)) for a in range(1, mx) for b in range(a, mx) if (c := sqrt(a**2 + b**2)).is_integer()]  # 利用赋值表达式防止重复计算斜边
print(triples)

- 字典推导式：`{key_expr: value_expr for item in iterable if condition}`。字典不允许重复键，重复键会被最后一个值覆盖。

In [ ]:
from string import ascii_lowercase

# 字典推导式
letter = {c:k for k, c in enumerate(ascii_lowercase, 1)}
print(letter)

# 字典推导式，其中包含生成器表达式
letter = dict((c, k) for k, c in enumerate(ascii_lowercase, 1))
print(letter)

# 字典推导式，注意字典键不能重复，否则会覆盖之前的值
# 输出：{'h': 'H', 'e': 'E', 'l': 'L', 'o': 'O'}
word = 'hello'
swaps = {c: c.swapcase() for c in word}
print(swaps)

- 集合推导式：`{expression for item in iterable if condition}`。集合自动去重，结果无重复元素。

In [ ]:
word = 'Hello'

# 集合推导式
# 注意集合是可变、无序、不重复的元素集合，所以输出元素顺序随机、元素不重复
letter1 = {c for c in word}
print(letter1)

letter2 = set(c for c in word)
print(letter1 == letter2)

### 3.生成器

- 生成器函数：与常规函数相似，只是在函数体中使用`yield`语句返回迭代器对象，每次调用`next()`方法时，从上次离开的地方继续执行，直到遇到`yield`语句。
    - 语法：
        ```python
        def generator():
            for item in iterable:
                yield item
        ```
    - 说明：
        - 生成器函数能节省内存空间，因为它只在需要时才生成下一个值，而不是一次生成所有值。
        - 生成器函数中可以使用`return`语句返回，将触发`StopIteration`异常。
    - 生成器对象方法：
        - `send(value)`：向生成器函数发送一个值，返回`yield`语句的表达式值。
        - `throw(type, value, traceback)`：向生成器函数抛出一个异常。
        - `close()`：关闭生成器对象，释放资源。
    - `yield from`语句：用于将一个可迭代对象的元素一个一个地`yield`给调用者，而不是返回一个包含所有元素的元器对象的值。
        - `yield from iterable`：类似于`for item in iterable: yield item`。


In [ ]:
# 定义一个生成器函数
def square(x):
    for i in range(x):
        yield i ** 2

sq = square(3)          # 创建一个生成器对象
print(sq)               # <generator object square at 0x...>
print(next(sq))         # 0
print(sq.__next__())    # 1，调用next()方法，实际上调用的是生成器对象的__next__()方法
print(next(sq))         # 4
# print(next(sq))       # 生成器已经耗尽，再调用next()会抛出StopIteration异常

In [ ]:
# 通过return语句触发StopIteration异常，结束生成器函数
def geometric(a, q):
    k = 0
    while True:
        result = a * q ** k
        if result <= 100000:
            yield result
        else:
            return
        k += 1

for i in geometric(2, 8):
    print(i)

In [ ]:
def counter(start=0):
    n = start
    while True:
        result = yield n
        print(type(result), result)
        if result == 'q':
            break
        n += 1

try:
    c = counter()                   # 创建生成器对象
    print(next(c))                  # 输出：0，获取第一个值
    print(c.send('Hello'))          # 输出：<class 'str'> Hello / 1，通过send()方法发送Hello，并返回yield语句对应表达式的值
    print(next(c))                  # 输出：<class 'NoneType'> None / 2，通过next()，yield返回None
    print(c.send('q'))              # 输出：<class 'str'> q，通过send()方法发送q，推出生成器，触发StopIteration异常
except StopIteration:
    print('StopIteration异常')
    c.close()

In [ ]:
# 利用yield from语句实现生成器函数
def squares2(start, end):
    yield from (x ** 2 for x in range(start, end))

for n in squares2(2, 5):
    print(n)

- 生成器表达式：`(表达式 for 变量 in 可迭代对象)`，与列表推导式非常类似，只是在表达式后面使用`()`括号。
    - 生成器表达式迭代一次就耗尽，再次迭代会触发`StopIteration`异常。
    - 在编写表达式的时候，要警惕额外的括号，否则会导致代码产生很大的区别。

In [ ]:
# 生成器表达式
cubes1 = ((x, x **3) for x in range(20) if x % 3 == 0 or x % 5 == 0)
print(list(cubes1))

# 利用map()和filter()函数实现生成器表达式
cubes2 = map(lambda x: (x, x ** 3), filter(lambda x: x % 3 == 0 or x % 5 == 0, range(20)))
print(list(cubes2))

### 4.注意事项

- 性能：在一般情况下，`map()`通常最快(C层面优化)，推导式和生成器表达式紧随其后，`for`循环最慢。
- 规则1：只需遍历一次就用生成器，需要多次访问或索引就用列表。
- 规则2：推导式超过2层嵌套或1行放不下时，改用`for`循环或生成器函数。
- 不要过度使用：可以尽可能地尝试使用推导式和生成器表达式。但是，如果代码开始变得复杂，就应该转换为更容易阅读的形式。
- 名称局部化：在推导式和生成器表达式中，变量的范围是局部的，不会影响到外部的变量。

### 5.本章小结

**核心知识脉络**

```text
推导式与生成器
│
├── map / zip / filter
│   ├── map(func, iter1, ...) → 迭代器，最短停止
│   ├── zip(iter1, iter2, ...) → 元组迭代器，最短停止
│   ├── filter(func, iter) → 保留 True 元素
│   └── filter(None, iter) → 去除 False 值 ⚠️
│
├── 推导式 ⭐
│   ├── 列表推导式 [expr for x in iter if cond]
│   ├── 字典推导式 {k:v for x in iter if cond}
│   ├── 集合推导式 {expr for x in iter if cond}
│   ├── 嵌套推导式（for 顺序 = 嵌套循环顺序）
│   └── 可读性边界（>2层改用函数/循环）
│
├── 生成器 ⭐
│   ├── 生成器函数（yield 暂停/恢复）
│   ├── 生成器表达式 (expr for x in iter if cond)
│   ├── yield from（委托子迭代器）
│   ├── send()（向生成器传值）
│   ├── throw()（在 yield 处抛异常）
│   ├── close()（关闭生成器）
│   └── 只能迭代一次 ⚠️
│
├── 性能对比
│   ├── 速度：map ≈ 推导式 > for 循环
│   ├── 内存：生成器 << 列表
│   └── 可读性 > 微优化
│
└── 名称局部化
    ├── 推导式变量 → 局部（不泄漏）✅
    └── for 循环变量 → 泄漏到外部 ⚠️
```

**四种推导式/生成器语法速查**

| 类型         | 语法                           | 结果类型 | 内存       | 可迭代次数   |
| ------------ | ------------------------------ | -------- | ---------- | ------------ |
| 列表推导式   | `[expr for x in iter if cond]` | `list`   | 高         | 无限次       |
| 字典推导式   | `{k:v for x in iter if cond}`  | `dict`   | 高         | 无限次       |
| 集合推导式   | `{expr for x in iter if cond}` | `set`    | 高         | 无限次       |
| 生成器表达式 | `(expr for x in iter if cond)` | 生成器   | **极低** ⭐ | **仅一次** ⚠️ |

**map/zip/filter速查**

| 函数     | 语法                     | 行为                 | 返回   |
| -------- | ------------------------ | -------------------- | ------ |
| `map`    | `map(func, iter1, ...)`  | 对每个元素应用func  | 迭代器 |
| `zip`    | `zip(iter1, iter2, ...)` | 配对为元组           | 迭代器 |
| `filter` | `filter(func, iter)`     | 保留func为True的元素 | 迭代器 |

**特性对比汇总**

| 特性       | for循环     | 列表推导式     | 生成器表达式     | 生成器函数       |
| ---------- | -------------- | -------------- | ---------------- | ---------------- |
| 内存占用   | 高（手动收集） | 高（完整列表） | **低（惰性）** ⭐ | **低（惰性）** ⭐ |
| 速度       | 最慢           | 快             | 快               | 快               |
| 可读性     | 高             | 高（简单时）   | 中               | 中               |
| 可迭代次数 | —              | 无限次         | **仅一次** ⚠️     | **仅一次** ⚠️     |
| 复杂逻辑   | ✅ 支持         | ⚠️ 有限         | ⚠️ 有限           | ✅ 支持           |
| 变量泄漏   | ⚠️ 会泄漏       | ✅ 不泄漏       | ✅ 不泄漏         | ✅ 不泄漏         |

**关键警告与提示**

| 类型   | 内容                                                         |
| ------ | ------------------------------------------------------------ |
| ⚠️ 警告 | `filter(None, seq)` 去除所有 falsy 值（0、False、''、[]等），不只是 None |
| ⚠️ 警告 | 生成器只能迭代一次，再次迭代得到空结果                       |
| ⚠️ 警告 | 嵌套推导式内层依赖外层变量时，必须写在外层 for 之后          |
| ⚠️ 警告 | `sum([n**2 for n in range(10**9)])` 会 MemoryError；`sum(n**2 for n in range(10**9))` 不会 ⭐ |
| ⚠️ 警告 | `send()` 首次必须用 `next(gen)` 或 `gen.send(None)` 启动     |
| ⚠️ 警告 | 字典推导式重复键会被最后一个值覆盖                           |
| ⚠️ 警告 | 集合推导式重复元素会被忽略                                 |
| 💡 技巧 | 只需遍历一次 → 生成器；需要多次访问或索引 → 列表             |
| 💡 技巧 | 推导式 > `map`/`filter` + `lambda`（更 Pythonic）            |
| 💡 技巧 | `yield from it` 替代 `for item in it: yield item`            |
| 💡 技巧 | 推导式超过 2 层嵌套 → 改用生成器函数或 `for` 循环            |
| 💡 技巧 | 函数参数中生成器表达式可省略外层括号：`sum(x**2 for x in range(10))` |